In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path("~/Documents/vggt").expanduser()))

import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 is supported on Ampere GPUs (Compute Capability 8.0+) 
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Initialize the model and load the pretrained weights.
# This will automatically download the model weights the first time it's run, which may take a while.
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)

# Load and preprocess example images (replace with your own image paths)
image_names = ["./data/frame_B0.jpg"]  
images = load_and_preprocess_images(image_names).to(device)

with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        # Predict attributes including cameras, depth maps, and point maps.
        predictions = model(images)

In [2]:
predictions.keys()

dict_keys(['pose_enc', 'pose_enc_list', 'depth', 'depth_conf', 'world_points', 'world_points_conf'])

In [8]:
import open3d as o3d
import matplotlib
matplotlib.use('Agg')
import numpy as np
import imageio
import matplotlib.pyplot as plt

# --- extract world points ---
world_points = predictions["world_points"].cpu().float().numpy()  # (S, H, W, 3)
conf = predictions["world_points_conf"].cpu().float().numpy()     # (S, H, W)

# flatten and filter by confidence
pts = world_points.reshape(-1, 3)
conf_flat = conf.reshape(-1)
pts = pts[conf_flat > 0.1]  # adjust threshold as needed

# --- save as PLY ---
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)
o3d.io.write_point_cloud("world_points.ply", pcd)
print(f"Saved {len(pts)} points to world_points.ply")

# --- render orbiting GIF ---
# --- offscreen render orbiting GIF ---
# images: (S, 3, H, W) -> (S, H, W, 3)
colors = images.cpu().float().permute(0, 2, 3, 1).numpy()
colors_flat = colors.reshape(-1, 3)[conf_flat > 0.1]

# downsample same indices
idx = np.random.choice(len(pts), size=min(10000, len(pts)), replace=False)
pts_sub = pts[idx]
colors_sub = colors_flat[idx]

frames = []
for angle in np.linspace(0, 360, 60, endpoint=False):
    fig = plt.figure(figsize=(6, 6), facecolor='black')
    ax = fig.add_subplot(111, projection='3d', facecolor='black')
    ax.scatter(pts_sub[:, 0], pts_sub[:, 2], -pts_sub[:, 1],
               s=0.5, c=colors_sub.clip(0, 1), alpha=0.6)
    ax.view_init(elev=0, azim=angle)
    ax.axis('off')
    plt.tight_layout(pad=0)

    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    frame = np.asarray(buf)[..., :3]
    frames.append(frame)
    plt.close(fig)

imageio.mimsave("world_points.gif", frames, fps=24, loop=0)
print("Saved world_points.gif")


Saved 268324 points to world_points.ply
Saved world_points.gif
